# NeuroSeek-MoE: ArXiv Healthcare+ML Paper Training Pipeline

Complete pipeline for training language models on healthcare+ML papers from ArXiv.

**Features:**
- ArXiv paper collection (30-40k papers) via NeMo Curator Pipeline API
- Direct LaTeX source extraction from S3 (cleaner than PDFs)
- NeMo Curator curation with custom healthcare stages (Linux/Colab only)
- Healthcare-specific preprocessing
- SentencePiece tokenizer training
- Model training with Colab-optimized settings
- Evaluation and inference export

**NeMo Curator Pipeline API:**
- Uses `download_arxiv()` to download directly from ArXiv (FREE, no AWS needed!)
- Custom `HealthcareFilterStage` for text cleaning and domain classification
- Custom `HealthcareQualityFilterStage` for deduplication and quality checks
- Custom `HealthcareJsonlWriter` for formatted output

**Requirements:**
- NeMo Curator: Requires Linux (Colab uses Linux, so it will work)
- No AWS credentials needed (uses direct ArXiv access, completely FREE)
- Timeline: 4-8 hours for 30-40k papers (depends on network speed)

**Note:** On non-Linux platforms, the pipeline falls back to basic preprocessing.


## 1. Setup Environment


In [ ]:
# Mount Google Drive (optional - for persistent storage)
from google.colab import drive
drive.mount('/content/drive')

# Set working directory
import os
WORK_DIR = '/content/neuroMOE'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"✅ Working directory: {WORK_DIR}")


In [ ]:
# Clone repository from GitHub
import sys
import subprocess

# Update this URL to your repository
REPO_URL = "https://github.com/oleeveeuh/neuroseekmoe.git"

# Clone the repository
if not os.path.exists(os.path.join(WORK_DIR, ".git")):
    print(f"📥 Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, "."], cwd=WORK_DIR, check=True)
    print("✅ Repository cloned successfully")
else:
    print("✅ Repository already exists, pulling latest changes...")
    subprocess.run(["git", "pull"], cwd=WORK_DIR, check=True)

# Add to path
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Verify essential files exist
required_files = ['data_pipeline.py', 'run_pipeline.py', 'config.yaml', 'train_colab.py']
missing_files = [f for f in required_files if not os.path.exists(os.path.join(WORK_DIR, f))]

if missing_files:
    print(f"⚠️  Warning: Missing files: {', '.join(missing_files)}")
    print("   Please check the repository URL and ensure all files are present")
else:
    print("✅ All required files present")


In [ ]:
# Install dependencies
print("📦 Installing dependencies...")

# Core dependencies
!pip install -q arxiv PyPDF2 sentencepiece pyyaml scikit-learn matplotlib pandas psutil

# PyTorch (Colab usually has it, but ensure it's available)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# DeepSpeed (optional, for advanced training features)
print("📦 Installing DeepSpeed (optional, for advanced training features)...")
try:
    !pip install -q deepspeed
    print("✅ DeepSpeed installed")
except Exception as e:
    print(f"⚠️  DeepSpeed installation failed (optional): {e}")
    print("   Training will continue without DeepSpeed")

# NeMo Curator (Linux/Colab only)
import platform
if platform.system() == 'Linux':
    print("🐧 Linux detected - installing NeMo Curator...")
    try:
        # Install NeMo Curator with text support
        # Use text_cuda12 for CUDA support, or [text] for CPU-only
        !pip install -q "nemo-curator[text_cuda12]"
        print("✅ NeMo Curator installed")
        
        # Note: No AWS credentials needed!
        print("\n✅ FREE Download - No AWS Required:")
        print("   The pipeline uses download_arxiv() which downloads directly from ArXiv")
        print("   This is completely FREE - no AWS account or charges!")
        print("   Timeline: 4-8 hours for 30-40k papers (depends on network speed)")
        
    except Exception as e:
        print(f"⚠️  NeMo Curator installation failed: {e}")
        print("   Pipeline will use fallback preprocessing")
else:
    print("⚠️  Non-Linux system - NeMo Curator not available")
    print("   Pipeline will use fallback preprocessing")

print("✅ Dependencies installed")


## 2. Configure Pipeline


In [ ]:
# Check if config.yaml exists, create/update if needed
import yaml
import platform

config_path = 'config.yaml'

if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("✅ Loaded existing config.yaml")
else:
    print("⚠️  config.yaml not found, using defaults")
    config = None

# Adjust config for Colab environment
if config:
    # Colab-optimized settings
    config['pipeline']['output_dir'] = './data/arxiv'
    config['training']['checkpoint_dir'] = './checkpoints'
    config['evaluation']['output_dir'] = './evaluations'
    config['inference']['output_dir'] = './inference'
    
    # Adjust for Colab GPU (T4, ~12GB VRAM)
    config['training']['batch_size'] = 6
    config['training']['gradient_accumulation_steps'] = 4
    config['training']['num_workers'] = 4
    
    # RAM-efficient ArXiv collection settings
    if 'collection' not in config:
        config['collection'] = {}
    config['collection']['batch_size'] = 10  # Papers per batch (RAM-efficient)
    config['collection']['ram_target'] = 50.0  # Target RAM percentage to stay below
    
    # NeMo Curator settings (only on Linux)
    if platform.system() == 'Linux':
        # Enable new Pipeline API (recommended, FREE - no AWS needed)
        config['nemo_curator']['use_pipeline_api'] = True
        config['nemo_curator']['raw_data_path'] = './data/arxiv/arxiv_raw_data'
        config['nemo_curator']['raw_output_path'] = './data/arxiv/arxiv_raw_output.jsonl'
        config['nemo_curator']['filter_query'] = 'cs.LG OR cs.AI OR q-bio.NC'  # Healthcare+ML
        config['nemo_curator']['max_workers'] = 1  # Colab safe, no rate limiting
        print("✅ NeMo Curator Pipeline API enabled (FREE - no AWS needed)")
        print("   📥 Will download directly from ArXiv using download_arxiv()")
        print("   ⏱️  Estimated time: 4-8 hours for 30-40k papers")
    else:
        print("⚠️  Non-Linux system - NeMo Curator will be skipped")
        # The pipeline will automatically use fallback preprocessing
    
    # Ensure max_papers is in both pipeline and collection for backward compatibility
    if 'pipeline' in config and 'max_papers' in config['pipeline']:
        if 'collection' not in config:
            config['collection'] = {}
        config['collection']['max_papers'] = config['pipeline']['max_papers']
    elif 'collection' in config and 'max_papers' in config['collection']:
        if 'pipeline' not in config:
            config['pipeline'] = {}
        config['pipeline']['max_papers'] = config['collection']['max_papers']
    
    # Save updated config
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print("✅ Config updated for Colab environment")
    
    # Display key settings
    print("\n📋 Key Configuration:")
    print(f"   Output directory: {config['pipeline']['output_dir']}")
    max_papers = config.get('pipeline', {}).get('max_papers') or config.get('collection', {}).get('max_papers', 30000)
    print(f"   Max papers: {max_papers}")
    print(f"   Collection batch size: {config['collection'].get('batch_size', 10)} papers/batch")
    print(f"   Collection RAM target: <{config['collection'].get('ram_target', 50.0)}%")
    print(f"   Training batch size: {config['training']['batch_size']}")
    print(f"   Max steps: {config['training']['max_steps']}")
    print(f"   Learning rate: {config['training']['learning_rate']}")


## 3. Run Complete Pipeline


In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = 'cuda'
else:
    print("⚠️  No GPU available - training will be slow")
    device = 'cpu'

# Set device for PyTorch
torch.cuda.empty_cache() if torch.cuda.is_available() else None


In [ ]:
# Run the complete pipeline
import subprocess
import sys
import os

print("🚀 Starting complete pipeline...")
print("=" * 80)

# Verify ArXiv package is installed
try:
    import arxiv
    print("✅ ArXiv package is available")
    
    # Test ArXiv API connectivity
    print("🔍 Testing ArXiv API connectivity...")
    try:
        client = arxiv.Client(page_size=1, delay_seconds=0.5, num_retries=1)
        search = arxiv.Search(query="cat:cs.LG", max_results=1, sort_by=arxiv.SortCriterion.SubmittedDate)
        test_result = next(client.results(search), None)
        if test_result:
            print("✅ ArXiv API is accessible")
        else:
            print("⚠️  ArXiv API returned no results (might be a query issue)")
    except Exception as e:
        print(f"⚠️  ArXiv API test failed: {e}")
        print("   This might indicate network or API issues")
except ImportError:
    print("❌ ArXiv package not found! Installing...")
    !pip install -q arxiv
    import arxiv
    print("✅ ArXiv package installed")

# Verify config exists
if not os.path.exists('config.yaml'):
    print("❌ config.yaml not found! Please run the configuration cell first.")
else:
    print("✅ Config file found")
    
    # Check for empty metadata file and delete it
    metadata_file = 'data/arxiv/arxiv_papers.jsonl'
    if os.path.exists(metadata_file):
        with open(metadata_file, 'r') as f:
            paper_count = sum(1 for line in f if line.strip())
        if paper_count == 0:
            print(f"⚠️  Found empty metadata file. Deleting to force fresh collection...")
            os.remove(metadata_file)
            print("✅ Empty file deleted")

# Run orchestration script
result = subprocess.run(
    [sys.executable, 'run_pipeline.py', '--config', 'config.yaml'],
    cwd=WORK_DIR,
    capture_output=False,
    text=True
)

if result.returncode == 0:
    print("\n✅ Pipeline completed successfully!")
    
    # Check if papers were actually collected
    metadata_file = 'data/arxiv/arxiv_papers.jsonl'
    if os.path.exists(metadata_file):
        with open(metadata_file, 'r') as f:
            paper_count = sum(1 for line in f if line.strip())
        print(f"\n📊 Papers collected: {paper_count}")
        
        if paper_count == 0:
            print("\n❌ ERROR: 0 papers collected!")
            print("   The pipeline will now fail with a clear error message.")
            print("   Possible issues:")
            print("   1. ArXiv API rate limiting or errors")
            print("   2. Network connectivity problems")
            print("   3. Query parameters too restrictive")
            print("   4. Check the collection logs above for error messages")
            print("\n   Try running the collection step manually to see detailed errors:")
            print("   !python data_pipeline.py collect --max-papers 1000")
    else:
        print(f"\n⚠️  WARNING: Metadata file not found at {metadata_file}")
else:
    print(f"\n❌ Pipeline failed with exit code {result.returncode}")
    print("   Check the logs above for details")


## 4. Monitor Progress (Optional)


In [ ]:
# Check pipeline status
import json
from pathlib import Path

report_path = Path('data/arxiv/pipeline_report.json')

if report_path.exists():
    with open(report_path, 'r') as f:
        report = json.load(f)
    
    print("📊 Pipeline Status Report")
    print("=" * 80)
    
    # Overall status
    pipeline_info = report.get('pipeline', {})
    print(f"\n⏱️  Total elapsed time: {pipeline_info.get('total_elapsed_hours', 0):.2f} hours")
    
    # Step status
    print("\n📦 Step Status:")
    steps = report.get('steps', {})
    for step_name, step_info in steps.items():
        status = "✅" if step_info.get('success') else "❌"
        elapsed = step_info.get('elapsed', 0)
        print(f"   {status} {step_name}: {elapsed:.2f}s")
    
    # Statistics
    stats = report.get('statistics', {})
    if stats:
        print("\n📊 Statistics:")
        if 'collected_papers' in stats:
            print(f"   Collected papers: {stats['collected_papers']}")
        if 'processed_papers' in stats:
            print(f"   Processed papers: {stats['processed_papers']}")
        if 'checkpoints' in stats:
            print(f"   Checkpoints: {stats['checkpoints']}")
        if 'latest_training_step' in stats:
            print(f"   Latest training step: {stats['latest_training_step']}")
else:
    print("⚠️  Pipeline report not found. Pipeline may still be running.")


In [ ]:
# Check training progress (if training has started)
import pandas as pd
import matplotlib.pyplot as plt

log_file = Path('checkpoints/training_log.csv')

if log_file.exists():
    df = pd.read_csv(log_file)
    
    if len(df) > 0:
        print(f"📈 Training Progress: {len(df)} log entries")
        
        # Plot loss curve
        if 'loss' in df.columns:
            plt.figure(figsize=(12, 4))
            
            plt.subplot(1, 2, 1)
            plt.plot(df['step'], df['loss'])
            plt.xlabel('Step')
            plt.ylabel('Loss')
            plt.title('Training Loss')
            plt.grid(True)
            
            if 'learning_rate' in df.columns:
                plt.subplot(1, 2, 2)
                plt.plot(df['step'], df['learning_rate'])
                plt.xlabel('Step')
                plt.ylabel('Learning Rate')
                plt.title('Learning Rate Schedule')
                plt.grid(True)
            
            plt.tight_layout()
            plt.show()
            
            # Display latest metrics
            print("\n📊 Latest Training Metrics:")
            latest = df.iloc[-1]
            print(f"   Step: {latest['step']}")
            print(f"   Loss: {latest['loss']:.4f}")
            if 'learning_rate' in latest:
                print(f"   Learning Rate: {latest['learning_rate']:.6f}")
            if 'gpu_memory_mb' in latest:
                print(f"   GPU Memory: {latest['gpu_memory_mb']:.0f} MB")
    else:
        print("⚠️  Training log is empty")
else:
    print("⚠️  Training log not found. Training may not have started yet.")


## 5. Resume Pipeline (If Interrupted)


In [ ]:
# Resume from a specific step if pipeline was interrupted
# Steps: 1=collect, 2=extract, 3=curate, 4=process, 5=tokenize, 6=train, 7=evaluate, 8=export

RESUME_FROM_STEP = None  # Set to step number (1-8) to resume from that step, or None to auto-detect

if RESUME_FROM_STEP:
    print(f"🔄 Resuming pipeline from step {RESUME_FROM_STEP}...")
    result = subprocess.run(
        [sys.executable, 'run_pipeline.py', '--config', 'config.yaml', '--start-from-step', str(RESUME_FROM_STEP)],
        cwd=WORK_DIR,
        capture_output=False,
        text=True
    )
    
    if result.returncode == 0:
        print("\n✅ Pipeline resumed successfully!")
    else:
        print(f"\n❌ Pipeline failed with exit code {result.returncode}")
else:
    print("ℹ️  Set RESUME_FROM_STEP to a number (1-8) to resume from that step")
    print("   Or run the pipeline normally - it will auto-detect and resume from the first incomplete step")


## 6. Download Results (Optional)


In [ ]:
# Download results to Google Drive or local machine
from google.colab import files
import shutil
from pathlib import Path

# Option 1: Download specific files
def download_file(file_path):
    """Download a file from Colab."""
    if os.path.exists(file_path):
        files.download(file_path)
        print(f"✅ Downloaded: {file_path}")
    else:
        print(f"⚠️  File not found: {file_path}")

# Option 2: Copy to Google Drive
def copy_to_drive(source_dir, drive_path='/content/drive/MyDrive/neuroMOE_results'):
    """Copy results to Google Drive."""
    os.makedirs(drive_path, exist_ok=True)
    
    if os.path.exists(source_dir):
        dest_path = os.path.join(drive_path, os.path.basename(source_dir))
        shutil.copytree(source_dir, dest_path, dirs_exist_ok=True)
        print(f"✅ Copied {source_dir} to {drive_path}")
    else:
        print(f"⚠️  Directory not found: {source_dir}")

# Example: Download tokenizer
# download_file('data/arxiv/healthcare_tokenizer.model')
# download_file('data/arxiv/healthcare_tokenizer.vocab')

# Example: Copy checkpoints to Drive
# copy_to_drive('checkpoints', '/content/drive/MyDrive/neuroMOE_results/checkpoints')

print("ℹ️  Uncomment the download/copy commands above to download results")


## Notes

- **NeMo Curator**: Only available on Linux (Colab uses Linux, so it will work). On other platforms, the pipeline automatically falls back to basic preprocessing.
- **GPU**: Colab provides free GPU (T4, ~12GB). The pipeline is optimized for this.
- **Storage**: Colab provides ~80GB disk space. For larger datasets, consider mounting Google Drive.
- **Time Limits**: Free Colab sessions have time limits. Use the resume feature to continue training.
- **Checkpoints**: Model checkpoints are saved every 5000 steps. You can resume from the latest checkpoint.

## Troubleshooting

- **Out of Memory**: Reduce `batch_size` or `max_papers` in config.yaml
- **NeMo Curator Errors**: The pipeline will automatically fall back to basic preprocessing
- **Interrupted Training**: Use the resume feature (Step 5) to continue from the last checkpoint
- **Slow Downloads**: ArXiv rate limiting is built-in. Be patient for large collections.
